# PacketGuard - Stage 1: Baseline + Model Comparison
NSL-KDD ingestion, preprocessing, and multi-model comparison (RF, SMOTE, Decision Tree, Voting).

In [ ]:
# Imports
import pandas as pd
import numpy as np
import joblib


In [ ]:
# NSL-KDD column names (file has no header row)
columns = [
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes",
    "land", "wrong_fragment", "urgent", "hot", "num_failed_logins",
    "logged_in", "num_compromised", "root_shell", "su_attempted",
    "num_root", "num_file_creations", "num_shells", "num_access_files",
    "num_outbound_cmds", "is_host_login", "is_guest_login", "count",
    "srv_count", "serror_rate", "srv_serror_rate", "rerror_rate",
    "srv_rerror_rate", "same_srv_rate", "diff_srv_rate",
    "srv_diff_host_rate", "dst_host_count", "dst_host_srv_count",
    "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate",
    "dst_host_serror_rate", "dst_host_srv_serror_rate",
    "dst_host_rerror_rate", "dst_host_srv_rerror_rate",
    "label", "difficulty"
]


In [ ]:
# Load train and test data
train_df = pd.read_csv("/Users/ananyadubey/project_data/KDDTrain+.txt", names=columns)
test_df = pd.read_csv("/Users/ananyadubey/project_data/KDDTest+.txt", names=columns)

print(train_df.shape, test_df.shape)
train_df.head()


In [ ]:
# Map raw attack labels to 5 standard categories
attack_map = {
    "normal": "Normal",

    # DoS
    "back": "DoS", "land": "DoS", "neptune": "DoS", "pod": "DoS",
    "smurf": "DoS", "teardrop": "DoS", "mailbomb": "DoS",
    "processtable": "DoS", "udpstorm": "DoS", "apache2": "DoS", "worm": "DoS",

    # Probe
    "satan": "Probe", "ipsweep": "Probe", "nmap": "Probe",
    "portsweep": "Probe", "mscan": "Probe", "saint": "Probe",

    # R2L
    "guess_passwd": "R2L", "ftp_write": "R2L", "imap": "R2L",
    "phf": "R2L", "multihop": "R2L", "warezmaster": "R2L",
    "warezclient": "R2L", "spy": "R2L", "xlock": "R2L", "xsnoop": "R2L",
    "snmpguess": "R2L", "snmpgetattack": "R2L", "httptunnel": "R2L",
    "sendmail": "R2L", "named": "R2L",

    # U2R
    "buffer_overflow": "U2R", "loadmodule": "U2R", "rootkit": "U2R",
    "perl": "U2R", "sqlattack": "U2R", "xterm": "U2R", "ps": "U2R",
}

def map_category(label):
    return attack_map.get(label.strip().lower(), "Unknown")


In [ ]:
# Apply category mapping to both train and test sets
train_df["attack_category"] = train_df["label"].apply(map_category)
test_df["attack_category"] = test_df["label"].apply(map_category)

print(train_df["attack_category"].value_counts())
print()
print(test_df["attack_category"].value_counts())


In [ ]:
# Split columns into categorical vs numeric
categorical_cols = ["protocol_type", "service", "flag"]
numeric_cols = [c for c in columns if c not in categorical_cols + ["label", "difficulty"]]

print(len(categorical_cols), len(numeric_cols))


In [ ]:
# Preprocessing imports
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler


In [ ]:
# Build the preprocessor: one-hot encode categoricals, scale numerics
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", StandardScaler(), numeric_cols),
    ]
)


In [ ]:
# Fit preprocessor on train only, transform both train and test
X_train_raw = train_df[categorical_cols + numeric_cols]
X_test_raw = test_df[categorical_cols + numeric_cols]

y_train_raw = train_df["attack_category"]
y_test_raw = test_df["attack_category"]

X_train = preprocessor.fit_transform(X_train_raw)
X_test = preprocessor.transform(X_test_raw)

print(X_train.shape, X_test.shape)


In [ ]:
# Encode target labels to integers
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train_raw)

print(label_encoder.classes_)


In [ ]:
# Encode test labels, drop any rows with categories unseen in train
known_mask = y_test_raw.isin(label_encoder.classes_)
print("Rows dropped (unseen category in test):", (~known_mask).sum())

X_test_final = X_test[known_mask.values]
y_test = label_encoder.transform(y_test_raw[known_mask])

print(X_test_final.shape, y_test.shape)


## Model 1: Random Forest (baseline)

In [ ]:
# Train: Random Forest (baseline, no resampling)
from sklearn.ensemble import RandomForestClassifier

clf = RandomForestClassifier(
    n_estimators=200,
    n_jobs=-1,
    random_state=42,
    class_weight="balanced"
)

clf.fit(X_train, y_train)


In [ ]:
# Test: predict on held-out KDDTest+
y_pred = clf.predict(X_test_final)


In [ ]:
# Eval: metrics imports
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)


In [ ]:
# Eval: Random Forest baseline metrics
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average="weighted", zero_division=0)
rec = recall_score(y_test, y_pred, average="weighted", zero_division=0)
f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1 (wtd):  {f1:.4f}")


In [ ]:
# Eval: per-class report
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_, zero_division=0))


In [ ]:
# Eval: confusion matrix
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm, index=label_encoder.classes_, columns=label_encoder.classes_)
print(cm_df)


In [ ]:
# Save baseline artifacts
import os
os.makedirs("models", exist_ok=True)


In [ ]:
# Save: preprocessor, RF baseline model, label encoder
joblib.dump(preprocessor, "models/preprocessor.joblib")
joblib.dump(clf, "models/rf_model.joblib")
joblib.dump(label_encoder, "models/label_encoder.joblib")

print("Saved.")


## Class balancing: SMOTE oversampling
Used as input to Models 2-5 below.

In [ ]:
# Install imbalanced-learn (SMOTE)
%pip install imbalanced-learn


In [ ]:
# Apply SMOTE to balance all 5 classes to the majority class size
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42, k_neighbors=3)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("Before:", pd.Series(y_train).value_counts().sort_index())
print("After:", pd.Series(y_train_res).value_counts().sort_index())


## Model 2: Random Forest + SMOTE + GridSearch

In [ ]:
# Train: Random Forest on SMOTE-resampled data, tuned via GridSearchCV
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators": [200, 300],
    "max_depth": [None, 30],
    "min_samples_split": [2, 5],
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42, class_weight="balanced", n_jobs=-1),
    param_grid, cv=3, scoring="accuracy", n_jobs=-1
)
grid.fit(X_train_res, y_train_res)

print(grid.best_params_)
clf_rf_smote = grid.best_estimator_


In [ ]:
# Test: predict on held-out KDDTest+
y_pred_rf_smote = clf_rf_smote.predict(X_test_final)


In [ ]:
# Eval: RF + SMOTE + GridSearch metrics
acc = accuracy_score(y_test, y_pred_rf_smote)
prec = precision_score(y_test, y_pred_rf_smote, average="weighted", zero_division=0)
rec = recall_score(y_test, y_pred_rf_smote, average="weighted", zero_division=0)
f1 = f1_score(y_test, y_pred_rf_smote, average="weighted", zero_division=0)

print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1 (wtd):  {f1:.4f}")
print(classification_report(y_test, y_pred_rf_smote, target_names=label_encoder.classes_, zero_division=0))


In [ ]:
# Eval: feature importance (why R2L/U2R stay low)
importances = pd.Series(clf_rf_smote.feature_importances_, index=preprocessor.get_feature_names_out())
print(importances.sort_values(ascending=False).head(20))


## Model 3: Decision Tree + SMOTE (best result)

In [ ]:
# Train: Decision Tree on SMOTE-resampled data
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(random_state=42, class_weight="balanced")
dt.fit(X_train_res, y_train_res)


In [ ]:
# Test: predict on held-out KDDTest+
y_pred_dt = dt.predict(X_test_final)


In [ ]:
# Eval: Decision Tree metrics
print("Decision Tree accuracy:", accuracy_score(y_test, y_pred_dt))
print(classification_report(y_test, y_pred_dt, target_names=label_encoder.classes_, zero_division=0))


## Model 4: Decision Tree + SMOTE + GridSearch (tuned, worse - overfit to CV)

In [ ]:
# Train: Decision Tree tuned via GridSearchCV
param_grid_dt = {
    "max_depth": [10, 20, 30, None],
    "min_samples_split": [2, 5, 10],
    "criterion": ["gini", "entropy"],
}

grid_dt = GridSearchCV(
    DecisionTreeClassifier(random_state=42, class_weight="balanced"),
    param_grid_dt, cv=3, scoring="accuracy", n_jobs=-1
)
grid_dt.fit(X_train_res, y_train_res)

print(grid_dt.best_params_)
dt_tuned = grid_dt.best_estimator_


In [ ]:
# Test: predict on held-out KDDTest+
y_pred_dt_tuned = dt_tuned.predict(X_test_final)


In [ ]:
# Eval: tuned Decision Tree metrics
print("Tuned Decision Tree accuracy:", accuracy_score(y_test, y_pred_dt_tuned))
print(classification_report(y_test, y_pred_dt_tuned, target_names=label_encoder.classes_, zero_division=0))


## Model 5: RF + DT Voting Ensemble

In [ ]:
# Train: hard-voting ensemble of RF + Decision Tree
from sklearn.ensemble import VotingClassifier

rf_orig = RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced", n_jobs=-1)
dt_orig = DecisionTreeClassifier(random_state=42, class_weight="balanced")

voting = VotingClassifier(estimators=[("rf", rf_orig), ("dt", dt_orig)], voting="hard")
voting.fit(X_train_res, y_train_res)


In [ ]:
# Test: predict on held-out KDDTest+
y_pred_vote = voting.predict(X_test_final)


In [ ]:
# Eval: voting ensemble metrics
print("Voting accuracy:", accuracy_score(y_test, y_pred_vote))
print(classification_report(y_test, y_pred_vote, target_names=label_encoder.classes_, zero_division=0))


## Multi-model comparison summary

In [ ]:
# Comparative analysis table across all models tried
comparison = pd.DataFrame([
    {"Method": "Random Forest (baseline)", "Accuracy": 0.7441, "Precision": 0.7996, "Recall": 0.7441, "F1": 0.6987, "R2L Recall": 0.02},
    {"Method": "RF + SMOTE + GridSearch", "Accuracy": 0.7401, "Precision": 0.8037, "Recall": 0.7401, "F1": 0.6965, "R2L Recall": 0.03},
    {"Method": "Decision Tree + SMOTE", "Accuracy": 0.7665, "Precision": 0.77, "Recall": 0.77, "F1": 0.74, "R2L Recall": 0.12},
    {"Method": "DT + SMOTE + GridSearch", "Accuracy": 0.7443, "Precision": 0.79, "Recall": 0.7443, "F1": 0.71, "R2L Recall": 0.08},
    {"Method": "RF + DT Voting Ensemble", "Accuracy": 0.7627, "Precision": 0.80, "Recall": 0.7627, "F1": 0.71, "R2L Recall": 0.00},
])

comparison = comparison.sort_values("Accuracy", ascending=False).reset_index(drop=True)
comparison


In [ ]:
# Bar chart: accuracy comparison across models
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))
plt.bar(comparison["Method"], comparison["Accuracy"], color="steelblue")
plt.ylabel("Accuracy")
plt.title("Model Comparison - NSL-KDD KDDTest+")
plt.xticks(rotation=30, ha="right")
plt.ylim(0.6, 0.85)
plt.tight_layout()
plt.show()


**Best result: Decision Tree + SMOTE - 76.65% accuracy**, R2L recall improved from 2% (RF baseline) to 12%.

Key finding: feature importance analysis shows all models rely heavily on traffic-volume
features (bytes, counts, service/flag) even after tuning. R2L attacks require login/content
features (num_failed_logins, root_shell, num_compromised) that exist in the dataset but are
consistently underweighted - indicating a data-level generalization ceiling rather than a
model/tuning limitation.

## Save best model (Decision Tree + SMOTE) for Stage 2

In [ ]:
# Save: preprocessor, best model (Decision Tree), label encoder
dt_final = DecisionTreeClassifier(random_state=42, class_weight="balanced")
dt_final.fit(X_train_res, y_train_res)

joblib.dump(preprocessor, "models/preprocessor.joblib")
joblib.dump(dt_final, "models/dt_model.joblib")
joblib.dump(label_encoder, "models/label_encoder.joblib")

print("Saved best model (Decision Tree + SMOTE) to models/dt_model.joblib")
